# 02. Detección del perro y clasificación de raza

Este notebook unifica el flujo de **detección del objeto perro** y **clasificación de raza** en un solo paso del pipeline.

La entrada oficial de este paso son las imágenes ya curadas en:

```text
images_curated_rgb_v2
```

La lógica del paso 02 es:

1. Cargar imágenes curadas.
2. Detectar si existe un perro usando YOLO.
3. Recortar la región del perro detectado.
4. Clasificar la raza usando un modelo CNN.
5. Calcular indicadores de detección, clasificación y estabilidad.
6. Guardar reportes finales para análisis posterior.

> Regla principal: la imagen no se clasifica directamente si primero no se valida la presencia del perro.

## Indicadores definidos para el paso 02

| Indicador | Descripción | Interpretación |
|---|---|---|
| Total de imágenes evaluadas | Número de imágenes curadas procesadas | Mide cobertura del paso 02 |
| Imágenes con perro detectado | Imágenes donde YOLO encontró la clase `dog` | Mide detección del objeto principal |
| Tasa de detección | Porcentaje de imágenes con perro detectado | Entre más alto, mejor cobertura |
| Imágenes sin perro detectado | Casos donde YOLO no detectó perro | Se mandan a revisión |
| Confianza promedio YOLO | Promedio de confianza de detecciones de perro | Mide seguridad del detector |
| Número promedio de perros detectados | Promedio de detecciones por imagen | Ayuda a identificar imágenes con múltiples perros |
| Imágenes clasificadas | Recortes enviados al clasificador | Debe coincidir con detecciones válidas |
| Accuracy | Porcentaje de razas correctamente clasificadas | Métrica principal del clasificador |
| Top-5 Accuracy | La raza correcta aparece en las 5 primeras predicciones | Importante para razas visualmente similares |
| Confianza promedio del clasificador | Probabilidad promedio de la raza predicha | Mide seguridad de la clasificación |
| Casos de baja confianza | Predicciones bajo un umbral definido | Sirven para revisión manual |
| Errores de procesamiento | Fallos técnicos durante detección, recorte o clasificación | Mide estabilidad del pipeline |

In [ ]:
# 0. Instalación de dependencias

!pip install -q ultralytics tensorflow opencv-python pillow scikit-learn pandas matplotlib tqdm

In [ ]:
# 1. Imports y configuración general

from pathlib import Path
import os
import json
import random
import time
import shutil

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm import tqdm
from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0, MobileNetV2
from tensorflow.keras.applications.efficientnet import preprocess_input as efficientnet_preprocess
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import regularizers

import ultralytics
from ultralytics import YOLO

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("Ultralytics:", ultralytics.__version__)

In [ ]:
!nvidia-smi

In [ ]:
from tensorflow.keras import mixed_precision

mixed_precision.set_global_policy("mixed_float16")
print("Mixed precision:", mixed_precision.global_policy())

In [ ]:
# 2. Montar Google Drive

from google.colab import drive

if Path("/content/drive/MyDrive").exists():
    print("Google Drive ya está montado.")
else:
    drive.mount("/content/drive")

In [ ]:
# 3. Rutas del proyecto

PROJECT_PATH = Path("/content/drive/MyDrive/proyecto_integrador")

# Entrada oficial del paso 02: imágenes curadas V2
CURATED_IMAGES_PATH = PROJECT_PATH / "images_curated_rgb_v2"

# Salidas del paso 02
DATA_PROCESSED_PATH = PROJECT_PATH / "data_processed"
STEP02_PATH = DATA_PROCESSED_PATH / "step02_detection_classification"
CROPS_PATH = STEP02_PATH / "dog_crops_from_curated_v2"
REPORTS_PATH = STEP02_PATH / "reports"
MODELS_PATH = PROJECT_PATH / "models"

for path in [DATA_PROCESSED_PATH, STEP02_PATH, CROPS_PATH, REPORTS_PATH, MODELS_PATH]:
    path.mkdir(parents=True, exist_ok=True)

DETECTION_REPORT_PATH = REPORTS_PATH / "step02_yolo_detection_report.csv"
CLASSIFICATION_REPORT_PATH = REPORTS_PATH / "step02_breed_classification_predictions.csv"
INDICATORS_PATH = REPORTS_PATH / "step02_indicators_summary.csv"
CLASS_NAMES_PATH = MODELS_PATH / "step02_class_names.json"

print("Proyecto:", PROJECT_PATH)
print("Entrada curada V2:", CURATED_IMAGES_PATH)
print("Crops:", CROPS_PATH)
print("Reportes:", REPORTS_PATH)
print("Modelos:", MODELS_PATH)

if not CURATED_IMAGES_PATH.exists():
    raise FileNotFoundError(
        f"No existe CURATED_IMAGES_PATH: {CURATED_IMAGES_PATH}. "
        "Primero ejecuta el notebook de curaduría V2 para crear images_curated_rgb_v2."
    )

## Regla de uso de imágenes para el paso 02

Para mantener consistencia con la curaduría visual, este notebook usa como entrada principal las imágenes de `images_curated_rgb_v2`.

La regla aplicada es:

| Caso | Decisión |
|---|---|
| YOLO detecta perro | Se recorta el perro y ese recorte se usa para clasificación |
| YOLO detecta múltiples perros | Se selecciona el bounding box más grande y/o de mayor confianza |
| YOLO no detecta perro | La imagen no se clasifica automáticamente y se marca como `needs_review` |
| Error técnico | Se registra en el reporte y se excluye del entrenamiento |
| Clasificador con baja confianza | Se conserva la predicción, pero se marca como caso de revisión |

Esta regla evita que el modelo de raza clasifique fondos, objetos irrelevantes o imágenes donde no se validó la presencia del perro.

In [ ]:
# 4. Construcción del dataframe base desde images_curated_rgb_v2

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def clean_breed_name(folder_name: str) -> str:
    """Convierte nombres tipo 'n02085620-Chihuahua' a 'Chihuahua'."""
    if "-" in folder_name:
        return folder_name.split("-", 1)[1].replace("_", " ")
    return folder_name.replace("_", " ")

records = []

for breed_folder in sorted(CURATED_IMAGES_PATH.iterdir()):
    if not breed_folder.is_dir():
        continue

    image_paths = [
        p for p in breed_folder.rglob("*")
        if p.suffix.lower() in IMAGE_EXTENSIONS
    ]

    for image_path in image_paths:
        records.append({
            "image_path": str(image_path),
            "image_name": image_path.name,
            "breed_folder": breed_folder.name,
            "breed": clean_breed_name(breed_folder.name)
        })

df = pd.DataFrame(records)

print("Total de imágenes curadas encontradas:", len(df))
print("Total de razas encontradas:", df["breed"].nunique())
df.head()

In [ ]:
# 5. Configuración del detector YOLO

YOLO_MODEL_NAME = "yolo26s.pt"   # Alternativas: "yolo26n.pt", "yolo26s.pt", "yolo11s.pt"
FALLBACK_YOLO_MODEL_NAME = "yolo11s.pt"

CONF_THRESHOLD = 0.25
# CROP_MARGIN = 0.08
# Mejora con más contexto
CROP_MARGIN = 0.15

def load_yolo_detector(primary_model: str, fallback_model: str):
    try:
        print(f"Intentando cargar detector: {primary_model}")
        model = YOLO(primary_model)
        selected_model = primary_model
    except Exception as e:
        print("No se pudo cargar el detector principal.")
        print("Error:", e)
        print(f"Usando detector de respaldo: {fallback_model}")
        model = YOLO(fallback_model)
        selected_model = fallback_model

    return model, selected_model

yolo_detector, selected_yolo_model = load_yolo_detector(
    YOLO_MODEL_NAME,
    FALLBACK_YOLO_MODEL_NAME
)

print("Detector seleccionado:", selected_yolo_model)
print("Clases disponibles:", yolo_detector.names)

dog_class_id = None
for class_id, class_name in yolo_detector.names.items():
    if class_name == "dog":
        dog_class_id = int(class_id)
        break

if dog_class_id is None:
    raise ValueError("No se encontró la clase 'dog' en el modelo YOLO seleccionado.")

print("ID de clase dog:", dog_class_id)

In [ ]:
# 6. Funciones de detección y recorte

def detect_dogs_yolo(image_path, model, dog_class_id, conf_threshold=0.25):
    """Detecta perros en una imagen usando YOLO."""
    results = model(str(image_path), conf=conf_threshold, verbose=False)

    detections = []

    for result in results:
        boxes = result.boxes

        if boxes is None:
            continue

        for box in boxes:
            class_id = int(box.cls[0])
            confidence = float(box.conf[0])

            if class_id == dog_class_id:
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
                area = max(0, x2 - x1) * max(0, y2 - y1)

                detections.append({
                    "class_id": class_id,
                    "class_name": "dog",
                    "confidence": confidence,
                    "x1": int(x1),
                    "y1": int(y1),
                    "x2": int(x2),
                    "y2": int(y2),
                    "area": int(area)
                })

    return detections


def select_best_detection(detections):
    """Selecciona la mejor detección priorizando área y confianza."""
    if len(detections) == 0:
        return None

    return sorted(
        detections,
        key=lambda d: (d["area"], d["confidence"]),
        reverse=True
    )[0]


def crop_detection(image_path, detection, output_path, margin=0.08):
    """Recorta el bounding box seleccionado y guarda el crop."""
    image = cv2.imread(str(image_path))

    if image is None:
        return None

    h, w = image.shape[:2]

    x1, y1, x2, y2 = detection["x1"], detection["y1"], detection["x2"], detection["y2"]

    box_w = x2 - x1
    box_h = y2 - y1

    mx = int(box_w * margin)
    my = int(box_h * margin)

    x1 = max(0, x1 - mx)
    y1 = max(0, y1 - my)
    x2 = min(w, x2 + mx)
    y2 = min(h, y2 + my)

    if x2 <= x1 or y2 <= y1:
        return None

    crop = image[y1:y2, x1:x2]

    output_path.parent.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(output_path), crop)

    return str(output_path)

In [ ]:
# 7. Prueba rápida con una imagen

sample_row = df.sample(1, random_state=SEED).iloc[0]
sample_path = Path(sample_row["image_path"])

detections = detect_dogs_yolo(
    sample_path,
    yolo_detector,
    dog_class_id,
    conf_threshold=CONF_THRESHOLD
)

print("Imagen:", sample_path)
print("Raza esperada:", sample_row["breed"])
print("Detecciones:", detections[:3])

# Visualización simple
img = cv2.imread(str(sample_path))
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(6, 6))
plt.imshow(img_rgb)
plt.title("Imagen curada V2 de ejemplo")
plt.axis("off")
plt.show()

In [ ]:
# 8. Procesamiento batch: detección + crops

# Para procesar todo el dataset, SAMPLE_LIMIT = None.
SAMPLE_LIMIT = None

SAVE_EVERY = 250
# FORCE_REPROCESS = False
# Mejora para regenerar los recortes
FORCE_REPROCESS = True


if SAMPLE_LIMIT is not None:
    process_df = df.sample(SAMPLE_LIMIT, random_state=SEED).reset_index(drop=True)
else:
    process_df = df.reset_index(drop=True)

print("Imágenes a procesar:", len(process_df))

# Cargar checkpoint si existe
processed_records = []
processed_paths = set()

if DETECTION_REPORT_PATH.exists() and not FORCE_REPROCESS:
    existing_df = pd.read_csv(DETECTION_REPORT_PATH)
    processed_records = existing_df.to_dict("records")
    processed_paths = set(existing_df["image_path"].astype(str).tolist())
    print("Checkpoint cargado:", len(processed_records), "registros")

start_time = time.time()

for idx, row in tqdm(process_df.iterrows(), total=len(process_df)):
    image_path = Path(row["image_path"])
    image_path_str = str(image_path)

    if image_path_str in processed_paths:
        continue

    breed_folder = row["breed_folder"]
    output_dir = CROPS_PATH / breed_folder
    output_path = output_dir / image_path.name

    record = {
        "image_path": image_path_str,
        "image_name": row["image_name"],
        "breed": row["breed"],
        "breed_folder": breed_folder,
        "selected_yolo_model": selected_yolo_model,
        "dog_detected": False,
        "num_detections": 0,
        "best_confidence": np.nan,
        "best_area": np.nan,
        "crop_path": None,
        "status": "not_processed",
        "error": None
    }

    try:
        if output_path.exists() and not FORCE_REPROCESS:
            record.update({
                "dog_detected": True,
                "num_detections": np.nan,
                "best_confidence": np.nan,
                "best_area": np.nan,
                "crop_path": str(output_path),
                "status": "crop_already_exists"
            })
        else:
            detections = detect_dogs_yolo(
                image_path,
                yolo_detector,
                dog_class_id,
                conf_threshold=CONF_THRESHOLD
            )

            best_detection = select_best_detection(detections)

            record["num_detections"] = len(detections)

            if best_detection is not None:
                crop_path = crop_detection(
                    image_path,
                    best_detection,
                    output_path,
                    margin=CROP_MARGIN
                )

                record.update({
                    "dog_detected": crop_path is not None,
                    "best_confidence": best_detection["confidence"],
                    "best_area": best_detection["area"],
                    "crop_path": crop_path,
                    "status": "detected_and_cropped" if crop_path is not None else "crop_failed"
                })
            else:
                record.update({
                    "dog_detected": False,
                    "status": "needs_review_no_dog_detected"
                })

    except Exception as e:
        record.update({
            "dog_detected": False,
            "status": "error",
            "error": str(e)
        })

    processed_records.append(record)
    processed_paths.add(image_path_str)

    if len(processed_records) % SAVE_EVERY == 0:
        pd.DataFrame(processed_records).to_csv(DETECTION_REPORT_PATH, index=False)

detection_df = pd.DataFrame(processed_records)
detection_df.to_csv(DETECTION_REPORT_PATH, index=False)

elapsed_minutes = (time.time() - start_time) / 60

print("Reporte guardado:", DETECTION_REPORT_PATH)
print("Tiempo total aproximado:", round(elapsed_minutes, 2), "minutos")
print(detection_df["dog_detected"].value_counts(dropna=False))
detection_df.head()

In [ ]:
PROJECT_PATH = Path("/content/drive/MyDrive/proyecto_integrador")
STEP02_PATH = PROJECT_PATH / "data_processed" / "step02_detection_classification"
REPORTS_PATH = STEP02_PATH / "reports"

DETECTION_REPORT_PATH = REPORTS_PATH / "step02_yolo_detection_report.csv"

if "detection_df" not in globals():
    if DETECTION_REPORT_PATH.exists():
        detection_df = pd.read_csv(DETECTION_REPORT_PATH)
        print("detection_df cargado desde:", DETECTION_REPORT_PATH)
        print("Registros:", len(detection_df))
    else:
        raise FileNotFoundError(
            f"No existe el reporte: {DETECTION_REPORT_PATH}. "
            "Primero corre la sección 8 de detección YOLO."
        )
else:
    print("detection_df ya existe en memoria.")
    print("Registros:", len(detection_df))

detection_df.head()

In [ ]:
!cp -r /content/drive/MyDrive/proyecto_integrador/data_processed/step02_detection_classification/dog_crops_from_curated_v2 /content/dog_crops_from_curated_v2

In [ ]:
# 9. Indicadores de detección YOLO

total_images = len(detection_df)
detected_images = int(detection_df["dog_detected"].sum())
not_detected_images = total_images - detected_images
detection_rate = detected_images / total_images * 100 if total_images > 0 else 0

avg_yolo_confidence = detection_df["best_confidence"].dropna().mean()
avg_num_detections = detection_df["num_detections"].dropna().mean()
error_count = int((detection_df["status"] == "error").sum())

detection_indicators = pd.DataFrame([
    {"indicator": "total_images_evaluated", "value": total_images},
    {"indicator": "images_with_dog_detected", "value": detected_images},
    {"indicator": "images_without_dog_detected", "value": not_detected_images},
    {"indicator": "dog_detection_rate_percent", "value": detection_rate},
    {"indicator": "avg_yolo_confidence", "value": avg_yolo_confidence},
    {"indicator": "avg_num_detections", "value": avg_num_detections},
    {"indicator": "processing_errors", "value": error_count},
    {"indicator": "selected_yolo_model", "value": selected_yolo_model},
])

detection_indicators

In [ ]:
# Visualización de indicadores de detección

plt.figure(figsize=(6, 4))
pd.Series({
    "Con perro detectado": detected_images,
    "Sin perro detectado": not_detected_images
}).plot(kind="bar")
plt.title("Resultado de detección del perro")
plt.ylabel("Cantidad de imágenes")
plt.xticks(rotation=0)
plt.show()

plt.figure(figsize=(6, 4))
detection_df["best_confidence"].dropna().plot(kind="hist", bins=30)
plt.title("Distribución de confianza YOLO")
plt.xlabel("Confianza")
plt.ylabel("Frecuencia")
plt.show()

## Lectura de los indicadores de detección

La tasa de detección indica qué porcentaje del dataset curado contiene una detección válida de perro.  
Las imágenes sin detección no necesariamente son incorrectas; pueden incluir perros pequeños, poses difíciles, fondos complejos o limitaciones del detector.

Estas imágenes se conservan como casos `needs_review` y no se usan automáticamente para entrenar el clasificador de raza.

In [ ]:
# 10. Preparación de datos para clasificación de raza


valid_crops_df = detection_df[
    (detection_df["dog_detected"] == True) &
    (detection_df["crop_path"].notna()) &
    (detection_df["status"].isin(["detected_and_cropped", "crop_already_exists"]))
].copy()

valid_crops_df["crop_path"] = valid_crops_df["crop_path"].astype(str)
valid_crops_df["breed"] = valid_crops_df["breed"].astype(str)

print("Crops válidos para clasificación:", len(valid_crops_df))
print("Razas disponibles:", valid_crops_df["breed"].nunique())

breed_counts = valid_crops_df["breed"].value_counts()
display(breed_counts.describe())

plt.figure(figsize=(14, 5))
breed_counts.head(40).plot(kind="bar")
plt.title("Top 40 razas con crops válidos")
plt.xlabel("Raza")
plt.ylabel("Cantidad de imágenes")
plt.xticks(rotation=90)
plt.show()

In [ ]:
# Mejora dos variantes
# 10B. Comparación de entrada: imagen completa vs crop YOLO

# Variante B: crop YOLO actual
crop_model_df = valid_crops_df[["crop_path", "breed"]].copy()
crop_model_df = crop_model_df.rename(columns={"crop_path": "image_input_path"})
crop_model_df["input_variant"] = "yolo_crop"

# Variante A: imagen completa curada V2
full_image_model_df = valid_crops_df[["image_path", "breed"]].copy()
full_image_model_df = full_image_model_df.rename(columns={"image_path": "image_input_path"})
full_image_model_df["input_variant"] = "full_curated_image"

print("Imágenes para variante crop YOLO:", len(crop_model_df))
print("Imágenes para variante imagen completa:", len(full_image_model_df))

display(crop_model_df.head())
display(full_image_model_df.head())

In [ ]:
# 11. División train / validation / test

# Para entrenar:
# "yolo_crop" usa el recorte del perro.
# "full_curated_image" usa la imagen completa curada V2.

INPUT_VARIANT = "yolo_crop"
# INPUT_VARIANT = "full_curated_image"

if INPUT_VARIANT == "yolo_crop":
    model_df = crop_model_df[["image_input_path", "breed"]].copy()
elif INPUT_VARIANT == "full_curated_image":
    model_df = full_image_model_df[["image_input_path", "breed"]].copy()
else:
    raise ValueError("INPUT_VARIANT debe ser 'yolo_crop' o 'full_curated_image'.")

print("Variante seleccionada:", INPUT_VARIANT)
print("Total de imágenes:", len(model_df))
print("Total de razas:", model_df["breed"].nunique())

In [ ]:
# 11B. Optimización de lectura: usar disco local de Colab

from pathlib import Path

if INPUT_VARIANT == "yolo_crop":
    DRIVE_BASE_PATH = "/content/drive/MyDrive/proyecto_integrador/data_processed/step02_detection_classification/dog_crops_from_curated_v2"
    LOCAL_BASE_PATH = "/content/dog_crops_from_curated_v2"

    # Copiar crops de YOLO desde Drive a disco local de Colab
    if not Path(LOCAL_BASE_PATH).exists():
        !cp -r "$DRIVE_BASE_PATH" "$LOCAL_BASE_PATH"

    # Cambiar rutas del dataframe
    model_df["image_input_path"] = model_df["image_input_path"].str.replace(
        DRIVE_BASE_PATH,
        LOCAL_BASE_PATH,
        regex=False
    )

elif INPUT_VARIANT == "full_curated_image":
    DRIVE_BASE_PATH = "/content/drive/MyDrive/proyecto_integrador/images_curated_rgb_v2"
    LOCAL_BASE_PATH = "/content/images_curated_rgb_v2"

    # Copiar imágenes completas curadas desde Drive a disco local de Colab
    if not Path(LOCAL_BASE_PATH).exists():
        !cp -r "$DRIVE_BASE_PATH" "$LOCAL_BASE_PATH"

    # Cambiar rutas del dataframe
    model_df["image_input_path"] = model_df["image_input_path"].str.replace(
        DRIVE_BASE_PATH,
        LOCAL_BASE_PATH,
        regex=False
    )

# Validar que los archivos existan
model_df["exists"] = model_df["image_input_path"].apply(lambda p: Path(p).exists())

print("Validación de archivos locales:")
print(model_df["exists"].value_counts())

# Quitar archivos que no existan, si hubiera alguno
model_df = model_df[model_df["exists"]].drop(columns=["exists"]).reset_index(drop=True)

print("Imágenes disponibles después de mover a /content:", len(model_df))
print(model_df["image_input_path"].head())

In [ ]:
# 11C. Filtrar clases con muy pocas imágenes para permitir stratify
min_images_per_class = 3
class_counts = model_df["breed"].value_counts()
valid_classes = class_counts[class_counts >= min_images_per_class].index

model_df = model_df[model_df["breed"].isin(valid_classes)].reset_index(drop=True)

CLASS_NAMES = sorted(model_df["breed"].unique())
NUM_CLASSES = len(CLASS_NAMES)

print("Imágenes para modelado:", len(model_df))
print("Número de clases:", NUM_CLASSES)

with open(CLASS_NAMES_PATH, "w") as f:
    json.dump(CLASS_NAMES, f, indent=2, ensure_ascii=False)

train_df, temp_df = train_test_split(
    model_df,
    test_size=0.30,
    random_state=SEED,
    stratify=model_df["breed"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["breed"]
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

In [ ]:
# 12. Generadores para EfficientNetB0

IMG_SIZE = 224
BATCH_SIZE = 64

'''
train_datagen_eff = ImageDataGenerator(
    preprocessing_function=efficientnet_preprocess,
    rotation_range=25,
    zoom_range=0.25,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.10,
    brightness_range=[0.75, 1.25],
    horizontal_flip=True,
    fill_mode="nearest"
) '''

# Mejora
train_datagen_eff = ImageDataGenerator(
    preprocessing_function=efficientnet_preprocess,
    rotation_range=15,
    zoom_range=0.15,
    width_shift_range=0.10,
    height_shift_range=0.10,
    shear_range=0.05,
    brightness_range=[0.85, 1.15],
    horizontal_flip=True,
    fill_mode="nearest"
)

eval_datagen_eff = ImageDataGenerator(
    preprocessing_function=efficientnet_preprocess
)

train_generator = train_datagen_eff.flow_from_dataframe(
    dataframe=train_df,
    # x_col="crop_path",
    x_col="image_input_path",
    y_col="breed",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    classes=CLASS_NAMES,
    shuffle=True,
    seed=SEED
)

val_generator = eval_datagen_eff.flow_from_dataframe(
    dataframe=val_df,
    # x_col="crop_path",
    x_col="image_input_path",
    y_col="breed",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    classes=CLASS_NAMES,
    shuffle=False
)

test_generator = eval_datagen_eff.flow_from_dataframe(
    dataframe=test_df,
    # x_col="crop_path",
    x_col="image_input_path",
    y_col="breed",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    classes=CLASS_NAMES,
    shuffle=False
)

In [ ]:
# 13. Modelo de clasificación de raza con EfficientNetB0

def build_efficientnetb0_model(num_classes, img_size=224):
    base_model = EfficientNetB0(
        weights="imagenet",
        include_top=False,
        input_shape=(img_size, img_size, 3)
    )

    base_model.trainable = False

    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = BatchNormalization()(x)
    x = Dense(
        512,
        activation="relu",
        kernel_regularizer=regularizers.l2(1e-4)
    )(x)
    x = Dropout(0.45)(x)
    x = Dense(
        256,
        activation="relu",
        kernel_regularizer=regularizers.l2(1e-4)
    )(x)
    x = Dropout(0.35)(x)
    outputs = Dense(num_classes, activation="softmax", dtype="float32")(x)

    model = Model(inputs=base_model.input, outputs=outputs)

    model.compile(
        optimizer=Adam(learning_rate=1e-4),
        loss="categorical_crossentropy",
        metrics=[
            "accuracy",
            tf.keras.metrics.TopKCategoricalAccuracy(k=5, name="top_5_accuracy")
        ]
    )

    return model, base_model

classifier_model, classifier_base = build_efficientnetb0_model(NUM_CLASSES, IMG_SIZE)
classifier_model.summary()

In [ ]:
# 14. Entrenamiento del clasificador

TRAIN_CLASSIFIER = True
EPOCHS_BASE = 40
EPOCHS_FINE_TUNE = 5

BEST_MODEL_PATH = MODELS_PATH / "step02_dog_breed_efficientnetb0_best.keras"
FINAL_MODEL_PATH = MODELS_PATH / "step02_dog_breed_efficientnetb0_final.keras"
TRAINING_LOG_PATH = REPORTS_PATH / "step02_training_log_efficientnetb0.csv"

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=BEST_MODEL_PATH,
        monitor="val_accuracy",
        save_best_only=True,
        mode="max",
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=2,
        min_lr=1e-7,
        verbose=1
    ),
    tf.keras.callbacks.CSVLogger(
        filename=TRAINING_LOG_PATH
    )
]

if TRAIN_CLASSIFIER:
    history_base = classifier_model.fit(
        train_generator,
        validation_data=val_generator,
        epochs=EPOCHS_BASE,
        callbacks=callbacks
    )

    classifier_base.trainable = True

    '''
    # Fine-tuning suave: solo últimas capas
    for layer in classifier_base.layers[:-40]:
        layer.trainable = False

    classifier_model.compile(
        optimizer=Adam(learning_rate=1e-5),
        loss="categorical_crossentropy",
        metrics=[
            "accuracy",
            tf.keras.metrics.TopKCategoricalAccuracy(k=5, name="top_5_accuracy")
        ]
    )
    '''

    # Mejora
    # Fine-tuning más conservador: solo últimas 25 capas
    for layer in classifier_base.layers[:-25]:
        layer.trainable = False

    classifier_model.compile(
        optimizer=Adam(learning_rate=3e-6),
        loss="categorical_crossentropy",
        metrics=[
            "accuracy",
            tf.keras.metrics.TopKCategoricalAccuracy(k=5, name="top_5_accuracy")
        ]
    )




    history_fine = classifier_model.fit(
        train_generator,
        validation_data=val_generator,
        epochs=EPOCHS_FINE_TUNE,
        callbacks=callbacks
    )

    classifier_model.save(FINAL_MODEL_PATH)
    print("Modelo final guardado en:", FINAL_MODEL_PATH)
else:
    if BEST_MODEL_PATH.exists():
        classifier_model = load_model(BEST_MODEL_PATH)
        print("Modelo cargado:", BEST_MODEL_PATH)
    else:
        raise FileNotFoundError(
            "TRAIN_CLASSIFIER=False pero no existe un modelo entrenado. "
            "Cambia TRAIN_CLASSIFIER=True o verifica BEST_MODEL_PATH."
        )

In [ ]:
# 15. Visualización del entrenamiento

def plot_history(history, title):
    plt.figure(figsize=(7, 4))
    plt.plot(history.history["accuracy"], label="train_accuracy")
    plt.plot(history.history["val_accuracy"], label="val_accuracy")
    plt.title(f"{title} - Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.show()

    plt.figure(figsize=(7, 4))
    plt.plot(history.history["loss"], label="train_loss")
    plt.plot(history.history["val_loss"], label="val_loss")
    plt.title(f"{title} - Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.show()

if TRAIN_CLASSIFIER:
    plot_history(history_base, "EfficientNetB0 Base")
    plot_history(history_fine, "EfficientNetB0 Fine-tuning")

In [ ]:
# 16. Evaluación en test

classifier_model = load_model(BEST_MODEL_PATH)

print("Modelo cargado para evaluación:", BEST_MODEL_PATH)

test_metrics = classifier_model.evaluate(test_generator, verbose=1)

test_loss = test_metrics[0]
test_accuracy = test_metrics[1]
test_top5_accuracy = test_metrics[2]

print("Test loss:", test_loss)
print("Test accuracy:", test_accuracy)
print("Test Top-5 accuracy:", test_top5_accuracy)

In [ ]:
# 17. Predicciones, Top-K y casos de baja confianza

pred_probs = classifier_model.predict(test_generator, verbose=1)
pred_indices = np.argmax(pred_probs, axis=1)
pred_confidence = np.max(pred_probs, axis=1)

index_to_class = {v: k for k, v in test_generator.class_indices.items()}
pred_labels = [index_to_class[i] for i in pred_indices]

true_labels = test_df["breed"].tolist()

TOP_K = 5
top_k_indices = np.argsort(pred_probs, axis=1)[:, -TOP_K:][:, ::-1]
top_k_labels = [
    [index_to_class[idx] for idx in row]
    for row in top_k_indices
]

predictions_df = test_df.copy().reset_index(drop=True)
predictions_df["predicted_breed"] = pred_labels
predictions_df["prediction_confidence"] = pred_confidence
predictions_df["correct"] = predictions_df["breed"] == predictions_df["predicted_breed"]
predictions_df["top_5_predictions"] = [json.dumps(labels, ensure_ascii=False) for labels in top_k_labels]
predictions_df["true_in_top_5"] = [
    true in labels
    for true, labels in zip(predictions_df["breed"], top_k_labels)
]

LOW_CONFIDENCE_THRESHOLD = 0.50
predictions_df["needs_review_low_confidence"] = (
    predictions_df["prediction_confidence"] < LOW_CONFIDENCE_THRESHOLD
)

predictions_df.to_csv(CLASSIFICATION_REPORT_PATH, index=False)

print("Reporte de predicciones guardado:", CLASSIFICATION_REPORT_PATH)
predictions_df.head()

In [ ]:
# 18. Indicadores finales del paso 02

classified_images = len(predictions_df)
classification_accuracy = predictions_df["correct"].mean() * 100
classification_top5_accuracy = predictions_df["true_in_top_5"].mean() * 100
avg_classifier_confidence = predictions_df["prediction_confidence"].mean()
low_confidence_cases = int(predictions_df["needs_review_low_confidence"].sum())


# Diagnóstico de desempeño en test
if classification_accuracy < 40 and classification_top5_accuracy < 70:
    test_diagnosis = "Posible subajuste: accuracy y Top-5 Accuracy son bajas en test."
elif classification_accuracy >= 70 and classification_top5_accuracy >= 90:
    test_diagnosis = "Buen desempeño general: accuracy y Top-5 Accuracy son sólidas para 120 clases."
elif classification_accuracy >= 60 and classification_top5_accuracy >= 85:
    test_diagnosis = "Desempeño aceptable: el modelo identifica patrones relevantes, aunque aún puede mejorar."
else:
    test_diagnosis = "Desempeño moderado: se recomienda revisar errores por raza y casos de baja confianza."


# Diagnóstico formal de sobreajuste/subajuste si existe log
overfit_underfit_diagnosis = "No calculado: requiere métricas de entrenamiento y validación."
generalization_gap = None
best_train_acc = None
best_val_acc = None
best_train_loss = None
best_val_loss = None
best_epoch = None

try:
    if TRAINING_LOG_PATH.exists():
        history_df = pd.read_csv(TRAINING_LOG_PATH)

        if {"accuracy", "val_accuracy", "loss", "val_loss"}.issubset(history_df.columns):
            best_val_idx = history_df["val_accuracy"].idxmax()
            best_row = history_df.loc[best_val_idx]

            best_epoch = int(best_val_idx + 1)
            best_train_acc = float(best_row["accuracy"])
            best_val_acc = float(best_row["val_accuracy"])
            best_train_loss = float(best_row["loss"])
            best_val_loss = float(best_row["val_loss"])

            generalization_gap = best_train_acc - best_val_acc

            if best_train_acc < 0.40 and best_val_acc < 0.40:
                overfit_underfit_diagnosis = (
                    "Posible subajuste: el modelo no aprende bien ni en entrenamiento ni en validación."
                )
            elif generalization_gap > 0.20:
                overfit_underfit_diagnosis = (
                    "Sobreajuste fuerte: entrenamiento supera demasiado a validación."
                )
            elif generalization_gap > 0.15:
                overfit_underfit_diagnosis = (
                    "Posible sobreajuste: existe una brecha importante entre entrenamiento y validación."
                )
            else:
                overfit_underfit_diagnosis = (
                    "Sin evidencia fuerte de sobreajuste o subajuste."
                )

except Exception as e:
    overfit_underfit_diagnosis = f"No calculado por error al leer training log: {e}"


# Consolidar indicadores
final_indicators = pd.DataFrame([
    {"section": "detection", "indicator": "total_images_evaluated", "value": total_images},
    {"section": "detection", "indicator": "images_with_dog_detected", "value": detected_images},
    {"section": "detection", "indicator": "images_without_dog_detected", "value": not_detected_images},
    {"section": "detection", "indicator": "dog_detection_rate_percent", "value": detection_rate},
    {"section": "detection", "indicator": "avg_yolo_confidence", "value": avg_yolo_confidence},
    {"section": "detection", "indicator": "avg_num_detections", "value": avg_num_detections},
    {"section": "detection", "indicator": "processing_errors", "value": error_count},
    {"section": "detection", "indicator": "selected_yolo_model", "value": selected_yolo_model},

    {"section": "classification", "indicator": "classified_images_test_set", "value": classified_images},
    {"section": "classification", "indicator": "test_accuracy_percent", "value": classification_accuracy},
    {"section": "classification", "indicator": "test_top5_accuracy_percent", "value": classification_top5_accuracy},
    {"section": "classification", "indicator": "avg_classifier_confidence", "value": avg_classifier_confidence},
    {"section": "classification", "indicator": "low_confidence_cases", "value": low_confidence_cases},
    {"section": "classification", "indicator": "test_diagnosis", "value": test_diagnosis},

    {"section": "generalization", "indicator": "best_epoch_by_val_accuracy", "value": best_epoch},
    {"section": "generalization", "indicator": "best_train_accuracy", "value": best_train_acc},
    {"section": "generalization", "indicator": "best_val_accuracy", "value": best_val_acc},
    {"section": "generalization", "indicator": "generalization_gap_train_minus_val", "value": generalization_gap},
    {"section": "generalization", "indicator": "best_train_loss", "value": best_train_loss},
    {"section": "generalization", "indicator": "best_val_loss", "value": best_val_loss},
    {"section": "generalization", "indicator": "overfit_underfit_diagnosis", "value": overfit_underfit_diagnosis},

    {"section": "stability", "indicator": "num_classes", "value": NUM_CLASSES},
    {"section": "stability", "indicator": "final_model_path", "value": str(FINAL_MODEL_PATH)},
])

final_indicators.to_csv(INDICATORS_PATH, index=False)

print("Indicadores guardados:", INDICATORS_PATH)
display(final_indicators)

In [ ]:
# Visualización de indicadores de clasificación

plt.figure(figsize=(6, 4))
pd.Series({
    "Accuracy": classification_accuracy,
    "Top-5 Accuracy": classification_top5_accuracy
}).plot(kind="bar")
plt.title("Desempeño del clasificador de raza")
plt.ylabel("Porcentaje")
plt.ylim(0, 100)
plt.xticks(rotation=0)
plt.show()

plt.figure(figsize=(6, 4))
predictions_df["prediction_confidence"].plot(kind="hist", bins=30)
plt.title("Distribución de confianza del clasificador")
plt.xlabel("Confianza")
plt.ylabel("Frecuencia")
plt.show()

plt.figure(figsize=(6, 4))
pd.Series({
    "Alta confianza": classified_images - low_confidence_cases,
    "Baja confianza": low_confidence_cases
}).plot(kind="bar")
plt.title("Casos de baja confianza")
plt.ylabel("Cantidad")
plt.xticks(rotation=0)
plt.show()

In [ ]:
# 19. Matriz de confusión

# Para 120 clases, una matriz completa puede ser difícil de leer.
# Mostramos las razas con más errores en el conjunto de test.

errors_df = predictions_df[predictions_df["correct"] == False].copy()

top_error_breeds = errors_df["breed"].value_counts().head(20)

plt.figure(figsize=(12, 5))
top_error_breeds.plot(kind="bar")
plt.title("Top 20 razas con más errores en test")
plt.xlabel("Raza")
plt.ylabel("Cantidad de errores")
plt.xticks(rotation=90)
plt.show()

display(errors_df.head(20))

In [ ]:
# 19B. Análisis de errores por raza

errors_df = predictions_df[predictions_df["correct"] == False].copy()

print("Total de errores:", len(errors_df))
print("Accuracy global:", round(predictions_df["correct"].mean() * 100, 2), "%")

# Razas con más errores
top_error_breeds = (
    errors_df["breed"]
    .value_counts()
    .reset_index()
)

top_error_breeds.columns = ["breed", "error_count"]

display(top_error_breeds.head(20))

plt.figure(figsize=(12, 5))
top_error_breeds.head(20).set_index("breed")["error_count"].plot(kind="bar")
plt.title("Top 20 razas con más errores")
plt.xlabel("Raza real")
plt.ylabel("Cantidad de errores")
plt.xticks(rotation=90)
plt.show()

# Confusiones más frecuentes
confusions = (
    errors_df
    .groupby(["breed", "predicted_breed"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

display(confusions.head(25))

In [ ]:
ERROR_ANALYSIS_PATH = REPORTS_PATH / "step02_error_analysis_by_breed.csv"
CONFUSIONS_PATH = REPORTS_PATH / "step02_frequent_confusions.csv"

top_error_breeds.to_csv(ERROR_ANALYSIS_PATH, index=False)
confusions.to_csv(CONFUSIONS_PATH, index=False)

print("Análisis de errores guardado en:", ERROR_ANALYSIS_PATH)
print("Confusiones frecuentes guardadas en:", CONFUSIONS_PATH)

In [ ]:
# 20. Función integrada para nuevas imágenes

def classify_image_with_classifier_only(image_path, classifier, class_names):
    """
    Clasifica una imagen usando únicamente el clasificador de raza.

    Esta función se usa cuando la imagen ya viene de image_input_path:
    - Si INPUT_VARIANT = "yolo_crop", image_input_path ya es el crop generado por YOLO.
    - Si INPUT_VARIANT = "full_curated_image", image_input_path es la imagen completa curada V2.
    """

    image_path = Path(image_path)

    img = tf.keras.utils.load_img(image_path, target_size=(IMG_SIZE, IMG_SIZE))
    arr = tf.keras.utils.img_to_array(img)
    arr = np.expand_dims(arr, axis=0)
    arr = efficientnet_preprocess(arr)

    probs = classifier.predict(arr, verbose=0)[0]

    pred_idx = int(np.argmax(probs))
    confidence = float(np.max(probs))

    top5_idx = np.argsort(probs)[-5:][::-1]
    top5 = [
        {
            "breed": class_names[int(i)],
            "confidence": float(probs[int(i)])
        }
        for i in top5_idx
    ]

    return {
        "image_path": str(image_path),
        "input_variant": INPUT_VARIANT,
        "predicted_breed": class_names[pred_idx],
        "prediction_confidence": confidence,
        "top_5_predictions": top5
    }


# Ejemplo con imagen de test
example_image = predictions_df.sample(1, random_state=SEED)["image_input_path"].iloc[0]

result = classify_image_with_classifier_only(
    example_image,
    classifier_model,
    CLASS_NAMES
)

result


In [ ]:
# 21. Comparación opcional YOLO11 vs YOLO26 en muestra


RUN_DETECTOR_COMPARISON = True
COMPARISON_SAMPLE_SIZE = 300

def benchmark_detector(model_name, sample_df):
    model = YOLO(model_name)

    dog_id = None
    for class_id, class_name in model.names.items():
        if class_name == "dog":
            dog_id = int(class_id)
            break

    if dog_id is None:
        raise ValueError(f"No se encontró clase dog en {model_name}")

    records = []
    start = time.time()

    for _, row in tqdm(sample_df.iterrows(), total=len(sample_df), desc=model_name):
        image_path = Path(row["image_path"])

        try:
            detections = detect_dogs_yolo(
                image_path,
                model,
                dog_id,
                conf_threshold=CONF_THRESHOLD
            )
            best = select_best_detection(detections)

            records.append({
                "model": model_name,
                "image_path": str(image_path),
                "dog_detected": best is not None,
                "num_detections": len(detections),
                "best_confidence": best["confidence"] if best else np.nan,
                "error": None
            })

        except Exception as e:
            records.append({
                "model": model_name,
                "image_path": str(image_path),
                "dog_detected": False,
                "num_detections": 0,
                "best_confidence": np.nan,
                "error": str(e)
            })

    elapsed = time.time() - start
    result_df = pd.DataFrame(records)

    summary = {
        "model": model_name,
        "sample_size": len(sample_df),
        "detection_rate_percent": result_df["dog_detected"].mean() * 100,
        "avg_confidence": result_df["best_confidence"].dropna().mean(),
        "images_without_detection": int((~result_df["dog_detected"]).sum()),
        "errors": int(result_df["error"].notna().sum()),
        "avg_seconds_per_image": elapsed / len(sample_df)
    }

    return result_df, summary

if RUN_DETECTOR_COMPARISON:
    sample_df = df.sample(
        min(COMPARISON_SAMPLE_SIZE, len(df)),
        random_state=SEED
    ).reset_index(drop=True)

    comparison_records = []
    detail_frames = []

    for model_name in ["yolo11s.pt", "yolo26s.pt"]:
        detail_df, summary = benchmark_detector(model_name, sample_df)
        detail_frames.append(detail_df)
        comparison_records.append(summary)

    detector_comparison_df = pd.DataFrame(comparison_records)
    detector_comparison_details_df = pd.concat(detail_frames, ignore_index=True)

    detector_comparison_df.to_csv(REPORTS_PATH / "step02_detector_comparison_summary.csv", index=False)
    detector_comparison_details_df.to_csv(REPORTS_PATH / "step02_detector_comparison_details.csv", index=False)

    display(detector_comparison_df)
else:
    print("Comparación opcional desactivada. Cambia RUN_DETECTOR_COMPARISON=True para ejecutarla.")

## Comparación experimental YOLO11s vs YOLO26s

Se realizó una comparación experimental entre YOLO11s y YOLO26s utilizando una muestra de 300 imágenes curadas. El objetivo fue evaluar qué detector era más conveniente para el paso 02 del pipeline, considerando tasa de detección, confianza promedio, imágenes sin detección, errores de procesamiento y tiempo promedio por imagen.

| Modelo | Imágenes evaluadas | Tasa de detección | Confianza promedio | Imágenes sin detección | Errores | Tiempo promedio por imagen |
|---|---:|---:|---:|---:|---:|---:|
| YOLO11s | 300 | 87.33% | 0.8039 | 38 | 0 | 1.9034 s |
| YOLO26s | 300 | 89.67% | 0.8062 | 31 | 0 | 0.0188 s |

Los resultados muestran que YOLO26s obtuvo mejor desempeño general. En comparación con YOLO11s, alcanzó una mayor tasa de detección, redujo el número de imágenes sin perro detectado de 38 a 31 y mantuvo cero errores de procesamiento. Además, presentó un tiempo promedio por imagen considerablemente menor.

Por esta razón, se selecciona YOLO26s como detector principal del paso 02. YOLO11s se conserva únicamente como alternativa de respaldo en caso de incompatibilidad del entorno.

# Análisis final del Paso 02: detección y clasificación de razas

En el Paso 02 se integraron dos tareas principales del pipeline: la detección del perro mediante YOLO y la clasificación de raza mediante EfficientNetB0. El objetivo fue validar primero la presencia del perro en la imagen, generar un recorte relevante y posteriormente clasificar la raza a partir de esa región visual.

## 1. Análisis de detección con YOLO

El modelo utilizado para la detección fue `yolo26s.pt`.

| Indicador | Resultado |
|---|---:|
| Imágenes evaluadas | 20,580 |
| Imágenes con perro detectado | 18,807 |
| Imágenes sin perro detectado | 1,773 |
| Tasa de detección | 91.38% |
| Confianza promedio de YOLO | 0.8110 |
| Promedio de detecciones por imagen | 1.0460 |
| Errores de procesamiento | 0 |

Los resultados muestran que YOLO26s detectó perros en **18,807 de 20,580 imágenes**, lo que representa una tasa de detección de **91.38%**. Este resultado indica que el detector logró identificar correctamente el objeto principal en la mayoría del dataset.

Las **1,773 imágenes sin detección** no necesariamente representan errores del dataset. Estos casos pueden deberse a perros pequeños, posturas difíciles, fondos complejos, baja visibilidad, recortes atípicos o limitaciones propias del detector.

Un resultado importante es que no se presentaron errores de procesamiento, lo que confirma que el flujo técnico de detección y recorte fue estable.

## 2. Análisis de clasificación de razas

Después de la detección, se entrenó un modelo EfficientNetB0 para clasificar la raza del perro usando las imágenes válidas generadas por YOLO.

| Indicador | Resultado |
|---|---:|
| Imágenes evaluadas en test | 2,822 |
| Número de clases | 120 |
| Test Accuracy | 77.14% |
| Test Top-5 Accuracy | 94.83% |
| Confianza promedio del clasificador | 0.7740 |
| Casos de baja confianza | 477 |

El clasificador alcanzó una **accuracy de prueba de 77.14%**, lo que significa que predijo correctamente la raza exacta en aproximadamente 77 de cada 100 imágenes del conjunto de prueba.

Además, obtuvo una **Top-5 Accuracy de 94.83%**. Esta métrica es especialmente importante en este proyecto porque se trata de una clasificación fina con **120 razas**, muchas de ellas visualmente similares. El resultado indica que, aunque el modelo no siempre predice la raza correcta como primera opción, en casi 95 de cada 100 casos la raza real aparece dentro de las cinco predicciones más probables.

La confianza promedio del clasificador fue de **0.7740**, lo cual indica un nivel razonable de seguridad en las predicciones. Sin embargo, se identificaron **477 casos de baja confianza**, los cuales pueden utilizarse para análisis posterior, revisión manual o mejora del dataset.

## 3. Análisis de desempeño general

El desempeño general del modelo puede considerarse sólido para la complejidad del problema. Clasificar **120 razas de perros** es una tarea difícil porque existen razas con características visuales muy parecidas, como tamaño, color de pelaje, forma de orejas, hocico, textura o postura.

El resultado de **77.14% de accuracy exacta** muestra que el modelo aprendió patrones visuales relevantes para diferenciar razas. Por otra parte, el **94.83% de Top-5 Accuracy** demuestra que el modelo suele ubicar la raza correcta dentro de sus principales alternativas, incluso cuando no acierta en la primera predicción.

Por esta razón, el desempeño se considera adecuado y defendible para un modelo de clasificación fina basado en imágenes.

## 4. Análisis de sobreajuste y subajuste

El diagnóstico automático indica:

| Indicador | Resultado |
|---|---:|
| Mejor epoch según validation accuracy | 1 |
| Train accuracy | 53.28% |
| Validation accuracy | 71.22% |
| Generalization gap | -17.93% |
| Train loss | 1.8756 |
| Validation loss | 1.1723 |
| Diagnóstico | Sin evidencia fuerte de sobreajuste o subajuste |

El modelo no presenta evidencia fuerte de sobreajuste. Normalmente, el sobreajuste ocurre cuando el desempeño en entrenamiento es mucho mayor que en validación. En este caso ocurre lo contrario: la accuracy de validación fue mayor que la accuracy de entrenamiento.

Esto puede explicarse por el uso de técnicas de regularización durante el entrenamiento, como augmentación de datos y Dropout. Durante entrenamiento, las imágenes se modifican artificialmente con transformaciones como rotación, zoom, desplazamientos o cambios de brillo, lo que hace que el aprendizaje sea más difícil. Además, Dropout reduce temporalmente la capacidad del modelo durante entrenamiento. En validación, estas transformaciones no se aplican y Dropout se desactiva, por lo que el desempeño puede ser mayor.

Tampoco se observa subajuste, ya que el modelo alcanzó una accuracy de prueba de **77.14%** y una Top-5 Accuracy de **94.83%**. Estos valores muestran que el modelo sí aprendió patrones visuales relevantes y logró generalizar adecuadamente.

## 5. Interpretación del Top-5 Accuracy

La clasificación de razas de perros es un problema de clasificación fina, ya que muchas razas comparten características visuales similares, como tamaño, color de pelaje, forma de orejas, hocico, postura o textura.

Por esta razón, además de la accuracy exacta, se utiliza **Top-5 Accuracy**.

Ejemplo:

Si la raza real es `Siberian husky` y el modelo predice:

1. Malamute  
2. Eskimo dog  
3. Siberian husky  
4. Samoyed  
5. Norwegian elkhound  

La predicción exacta no sería correcta, pero la raza real sí aparece dentro de las cinco opciones más probables. En este caso, cuenta como acierto para Top-5 Accuracy.

El resultado de **94.83% en Top-5 Accuracy** indica que el modelo aprendió representaciones visuales útiles y logra ubicar la raza correcta entre sus principales alternativas en la gran mayoría de los casos.

## 6. Umbrales usados para interpretar sobreajuste y subajuste

Para este proyecto se utilizaron criterios prácticos de interpretación:

| Situación | Criterio |
|---|---|
| Posible subajuste | Train accuracy y validation accuracy menores a 40% |
| Subajuste fuerte | Accuracy baja y Top-5 Accuracy menor a 70% |
| Posible sobreajuste | Train accuracy supera validation accuracy por más de 15 puntos |
| Sobreajuste fuerte | Train accuracy supera validation accuracy por más de 20 puntos y validation loss aumenta |
| Sin evidencia fuerte | Brecha controlada entre entrenamiento y validación, con desempeño adecuado en test |

Con base en estos criterios, el modelo no muestra subajuste porque sus métricas de prueba son sólidas. Tampoco muestra sobreajuste porque no existe una brecha donde el entrenamiento sea mucho mejor que la validación.

## Conclusión final del Paso 02

El Paso 02 integró exitosamente la detección del perro y la clasificación de raza en un solo flujo de visión computacional.

Primero, YOLO26s detectó perros en **18,807 de 20,580 imágenes**, alcanzando una tasa de detección de **91.38%** con una confianza promedio de **0.8110** y sin errores de procesamiento. Esto permitió generar entradas visuales más relevantes para el clasificador al enfocarse en la región donde se encuentra el perro.

Posteriormente, EfficientNetB0 logró una **accuracy de prueba de 77.14%** y una **Top-5 Accuracy de 94.83%** sobre **120 razas**. Estos resultados son sólidos considerando que se trata de una tarea de clasificación fina, donde muchas clases comparten rasgos visuales similares.

El análisis de generalización no muestra evidencia fuerte de sobreajuste ni de subajuste. La validación fue superior al entrenamiento, lo cual es consistente con el uso de augmentación y Dropout durante la fase de entrenamiento. Además, el desempeño en test confirma que el modelo aprendió patrones visuales útiles y generaliza adecuadamente.

En conclusión, el Paso 02 produce un modelo funcional, estable y defendible para clasificación de razas. También genera salidas clave para los siguientes pasos del proyecto: extracción de embeddings, comparación visual mediante similitud coseno y análisis de similitud entre perros de la misma raza.